In [ ]:
# Cell 1 — Initialize the remote Colab runtime from files uploaded through VS Code

from pathlib import Path
import subprocess
import sys
import os
import shutil
import json
import hashlib
import random
import csv
import copy
import time
import zipfile
from datetime import datetime, timezone

UPLOAD_DIR = Path("/content/colab_upload")
CODE_BUNDLE = UPLOAD_DIR / "colab_code_bundle.zip"
UPLOADED_DATASET_DIR = UPLOAD_DIR / "datasets"

if not CODE_BUNDLE.is_file():
    raise FileNotFoundError(
        f"{CODE_BUNDLE} not found. In VS Code Explorer, right-click the local "
        "'colab_upload' folder and choose 'Upload to Colab' after connecting the Colab kernel."
    )
if not (UPLOADED_DATASET_DIR / "annotations" / "fold_manifest.csv").is_file():
    raise FileNotFoundError(
        "The uploaded colab_upload/datasets folder does not contain "
        "annotations/fold_manifest.csv. Run scripts/prepare_colab_upload.py locally, "
        "then upload the updated colab_upload folder."
    )

PROJECT_DIR = Path("/content/ParkinsonAiResearch")
if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(CODE_BUNDLE, "r") as archive:
    archive.extractall(PROJECT_DIR)

shutil.copytree(
    UPLOADED_DATASET_DIR,
    PROJECT_DIR / "datasets",
    dirs_exist_ok=True,
)

sys.path.insert(0, str(PROJECT_DIR))
os.chdir(PROJECT_DIR)

MANIFEST_PATH = PROJECT_DIR / "datasets" / "annotations" / "fold_manifest.csv"
OUTPUT_DIR = PROJECT_DIR / "outputs" / "wideresnet_colab"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Remote project root:", PROJECT_DIR)
print("Uploaded dataset:", PROJECT_DIR / "datasets")
print("Manifest:", MANIFEST_PATH)
print("Output directory:", OUTPUT_DIR)
print("Code folders:", [p.name for p in PROJECT_DIR.iterdir() if p.is_dir()])

In [ ]:
# Cell 2 — Install only notebook-side analysis dependencies.
# Colab normally provides PyTorch and Torchvision; we do not reinstall them here.

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "pandas", "matplotlib", "scikit-learn"],
    check=True
)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import models
from torch.utils.data import DataLoader

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    print("WARNING: No CUDA GPU detected. In Colab, select Runtime > Change runtime type > GPU.")

In [ ]:
# Cell 3 — Verify uploaded fold package and remote GPU

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "The selected Colab runtime has no CUDA GPU. In VS Code, use Select Kernel > Colab "
        "> New Colab Server and choose a GPU runtime if available."
    )

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)
print("GPU memory (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

if not (PROJECT_DIR / "config" / "__init__.py").is_file():
    raise FileNotFoundError("Uploaded code bundle is incomplete: config/__init__.py is missing.")
if not (PROJECT_DIR / "data" / "dataloader.py").is_file():
    raise FileNotFoundError("Uploaded code bundle is incomplete: data/dataloader.py is missing.")
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(f"Fold manifest not found: {MANIFEST_PATH}")

print("Code bundle:", "OK")
print("Fold manifest:", "OK")
print("Dataset raw folder:", (PROJECT_DIR / "datasets" / "raw").is_dir())
print("Runtime working directory:", Path.cwd())

In [ ]:
# Cell 4 — Experiment configuration (edit these values if needed)

FOLD_ID = 1
SEED = 42
IMAGE_SIZE = 224
BATCH_SIZE = 4
EPOCHS = 15
NUM_WORKERS = 2

PRETRAINED = True
FREEZE_BACKBONE = True
EMBEDDING_DIM = 64

HEAD_LR = 3e-4
BACKBONE_LR = 1e-5
WEIGHT_DECAY = 1e-4
FOCAL_GAMMA = 2.0

LABEL_SOURCE = "consensus"  # finalized three-rater expert labels
CLASS_NAMES = ("control", "parkinson", "unknown")
CLASS_TO_INDEX = {name: i for i, name in enumerate(CLASS_NAMES)}

# All experiments use the same fold, seed, model architecture and training budget.
EXPERIMENTS = ["cross_entropy", "weighted_cross_entropy", "focal"]

USE_AMP = torch.cuda.is_available()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if DEVICE.type != "cuda":
    print("WARNING: training will run on CPU. Select a GPU runtime for practical experiments.")
else:
    print("Training device:", DEVICE)

print({
    "fold": FOLD_ID,
    "seed": SEED,
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "pretrained": PRETRAINED,
    "freeze_backbone": FREEZE_BACKBONE,
    "embedding_dim": EMBEDDING_DIM,
    "losses": EXPERIMENTS,
    "mixed_precision": USE_AMP,
})

In [ ]:
# Cell 5 — Validate the fold-specific, development-only upload package

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

SOURCE_MANIFEST_SHA256 = sha256_file(MANIFEST_PATH)
UPLOAD_METADATA_PATH = UPLOAD_DIR / "upload_manifest.json"
if not UPLOAD_METADATA_PATH.is_file():
    raise FileNotFoundError("upload_manifest.json is missing; rebuild the local colab_upload package.")
upload_metadata = json.loads(UPLOAD_METADATA_PATH.read_text(encoding="utf-8"))
if int(upload_metadata.get("fold_id", -1)) != FOLD_ID:
    raise ValueError(
        f"Uploaded package is for Fold {upload_metadata.get('fold_id')}, "
        f"but this notebook is configured for Fold {FOLD_ID}. Rebuild and upload the matching package."
    )
if int(upload_metadata.get("test_images_included", -1)) != 0:
    raise RuntimeError("The staged upload package must not contain Test images.")

with MANIFEST_PATH.open("r", encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)
    manifest_columns = list(reader.fieldnames or [])
    manifest_rows = list(reader)

required = {
    "development_model_fold", "drawing_id", "participant_id", "cohort",
    "image_name", "image_path", "role", "cv_eligible", "expert_label",
    "label_status", "partition", "workbook_fold_id"
}
missing_columns = required - set(manifest_columns)
if missing_columns:
    raise ValueError(f"Manifest missing required columns: {sorted(missing_columns)}")
if not manifest_rows:
    raise ValueError("Uploaded fold manifest is empty.")

FINAL_LABEL_STATUSES = {
    "finalized_by_unanimous_vote",
    "finalized_by_control_majority",
    "finalized_as_unknown",
}

def is_true(value):
    return str(value).strip().lower() in {"true", "1", "yes", "y"}

fold_values = {str(row["development_model_fold"]).strip() for row in manifest_rows}
if fold_values != {str(FOLD_ID)}:
    raise ValueError(
        f"Expected only Fold {FOLD_ID} in the staged manifest; found {sorted(fold_values)}"
    )

if any(row.get("role") == "test_locked" or row.get("partition") == "heldout_test"
       for row in manifest_rows):
    raise RuntimeError("Test rows must not be included in this Colab upload package.")

expected_roles = {"train", "validation"}
actual_roles = {str(row.get("role", "")).strip() for row in manifest_rows}
if not actual_roles.issubset(expected_roles):
    raise ValueError(f"Unexpected roles in staged manifest: {actual_roles}")

missing_images = []
for row in manifest_rows:
    if not is_true(row.get("cv_eligible", "")):
        raise ValueError(f"Staged manifest contains non-eligible row: {row.get('drawing_id')}")
    if row.get("label_status") not in FINAL_LABEL_STATUSES:
        raise ValueError(f"Staged manifest contains non-final label: {row.get('drawing_id')}")
    if row.get("partition") != "development":
        raise ValueError(f"Non-development row found: {row.get('drawing_id')}")
    if row.get("expert_label", "").strip().lower() not in {"control", "parkinson", "unknown"}:
        raise ValueError(f"Unsupported expert label: {row.get('expert_label')}")

    image_path = (PROJECT_DIR / row["image_path"]).resolve()
    if not image_path.is_file():
        missing_images.append((row.get("drawing_id"), str(image_path)))

if missing_images:
    display(pd.DataFrame(missing_images, columns=["drawing_id", "missing_path"]).head(10))
    raise FileNotFoundError(f"{len(missing_images)} staged images are missing.")

train_subjects = {
    row["participant_id"] for row in manifest_rows if row["role"] == "train"
}
val_subjects = {
    row["participant_id"] for row in manifest_rows if row["role"] == "validation"
}
if train_subjects & val_subjects:
    raise RuntimeError(f"Participant leakage in staged package: {train_subjects & val_subjects}")

drawing_ids = [row["drawing_id"] for row in manifest_rows]
if len(drawing_ids) != len(set(drawing_ids)):
    raise ValueError("Duplicate drawing_id values in staged manifest.")

TEMP_MANIFEST_PATH = MANIFEST_PATH

print("Manifest SHA-256:", SOURCE_MANIFEST_SHA256)
print("Rows:", len(manifest_rows))
print("Train rows:", sum(row["role"] == "train" for row in manifest_rows))
print("Validation rows:", sum(row["role"] == "validation" for row in manifest_rows))
print("Train subjects:", len(train_subjects))
print("Validation subjects:", len(val_subjects))
print("Test rows/images included: 0")
print("All staged image paths exist.")

In [ ]:
# Cell 6 — Import project data pipeline and build train/validation loaders

import sys
sys.path.insert(0, str(PROJECT_DIR))

from config import PreprocessingConfig
from data.dataloader import (
    make_loader,
    build_train_transform,
    build_eval_transform,
)

preprocess_config = PreprocessingConfig(image_size=IMAGE_SIZE)
train_transform = build_train_transform(preprocess_config)
eval_transform = build_eval_transform(preprocess_config)

def make_fold_loaders():
    common = dict(
        fold_id=FOLD_ID,
        manifest_path=TEMP_MANIFEST_PATH,
        root=PROJECT_DIR,
        label_source=LABEL_SOURCE,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        seed=SEED,
    )
    train_loader = make_loader(
        split="train",
        transform=train_transform,
        **common,
    )
    val_loader = make_loader(
        split="val",
        transform=eval_transform,
        allow_empty=True,
        **common,
    )
    return train_loader, val_loader

train_loader, val_loader = make_fold_loaders()

if len(train_loader.dataset) == 0:
    raise ValueError("Training split is empty.")
if len(val_loader.dataset) == 0:
    raise ValueError(
        f"Fold {FOLD_ID} has no eligible validation images. "
        "Choose a fold with validation samples for this pilot; do not invent a validation split."
    )

train_subjects = {r["participant_id"] for r in train_loader.dataset.rows}
val_subjects = {r["participant_id"] for r in val_loader.dataset.rows}
assert train_subjects.isdisjoint(val_subjects), "Participant leakage between train and validation!"

print("Train images:", len(train_loader.dataset))
print("Train subjects:", len(train_subjects))
print("Train labels:", train_loader.dataset.label_counts())
print("Validation images:", len(val_loader.dataset))
print("Validation subjects:", len(val_subjects))
print("Validation labels:", val_loader.dataset.label_counts())
print("Train/validation subject overlap:", train_subjects & val_subjects)
print("Test rows loaded: 0")

In [ ]:
# Cell 7 — Visual quality control after the current deterministic transform

def show_batch(loader, transform, n=8):
    batch = next(iter(loader))
    n = min(n, len(batch["image"]))
    fig, axes = plt.subplots(2, (n + 1) // 2, figsize=(14, 6))
    axes = np.array(axes).reshape(-1)
    for i in range(n):
        image = transform.preview(batch["image"][i])
        axes[i].imshow(image)
        axes[i].set_title(
            f"{batch['label_name'][i]}\\n{batch['participant_id'][i]}",
            fontsize=8
        )
        axes[i].axis("off")
    for j in range(n, len(axes)):
        axes[j].axis("off")
    plt.tight_layout()
    plt.show()

show_batch(train_loader, train_transform, n=8)

In [ ]:
# Cell 8 — Focal Loss and train-fold-only class weights

class FocalLoss(nn.Module):
    """Multiclass focal loss with optional per-class alpha weights."""

    def __init__(self, alpha=None, gamma=2.0, reduction="mean"):
        super().__init__()
        if gamma < 0:
            raise ValueError("gamma must be non-negative")
        if reduction not in {"mean", "sum", "none"}:
            raise ValueError("reduction must be mean, sum, or none")
        alpha_tensor = None if alpha is None else torch.as_tensor(alpha, dtype=torch.float32)
        if alpha_tensor is not None and alpha_tensor.ndim != 1:
            raise ValueError("alpha must be a one-dimensional vector")
        self.register_buffer("alpha", alpha_tensor)
        self.gamma = float(gamma)
        self.reduction = reduction

    def forward(self, logits, targets):
        if logits.ndim != 2:
            raise ValueError("logits must have shape [batch, classes]")
        if targets.ndim != 1 or targets.shape[0] != logits.shape[0]:
            raise ValueError("targets must have shape [batch]")
        if targets.dtype != torch.long:
            raise TypeError("targets must be torch.long")
        if targets.numel() and (targets.min() < 0 or targets.max() >= logits.shape[1]):
            raise ValueError("target class index is out of range")

        log_probs = F.log_softmax(logits, dim=1)
        log_pt = log_probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        pt = log_pt.exp()
        loss = -((1.0 - pt).clamp_min(0.0) ** self.gamma) * log_pt

        if self.alpha is not None:
            if self.alpha.numel() != logits.shape[1]:
                raise ValueError("alpha length must equal the number of classes")
            alpha = self.alpha.to(device=logits.device, dtype=logits.dtype)
            loss = loss * alpha[targets]

        if self.reduction == "sum":
            return loss.sum()
        if self.reduction == "none":
            return loss
        return loss.mean()


def train_fold_class_weights(label_counts, class_names=CLASS_NAMES):
    counts = torch.tensor(
        [int(label_counts.get(name, 0)) for name in class_names],
        dtype=torch.float32
    )
    if (counts < 0).any() or counts.sum() == 0:
        raise ValueError("Invalid or empty train-fold class counts")
    present = counts > 0
    weights = torch.ones_like(counts)
    weights[present] = counts[present].sum() / (
        present.sum().item() * counts[present]
    )
    weights[present] = weights[present] / weights[present].mean()
    return weights


train_counts = train_loader.dataset.label_counts()
CLASS_WEIGHTS = train_fold_class_weights(train_counts)

print("Train-only class counts:", train_counts)
print("Class order:", CLASS_NAMES)
print("Weights:", CLASS_WEIGHTS.tolist())

for class_name in CLASS_NAMES:
    if train_counts.get(class_name, 0) == 0:
        print(f"WARNING: {class_name} has no training examples in this fold.")

In [ ]:
# Cell 9 — Unit checks for the Focal Loss implementation

torch.manual_seed(7)
test_logits = torch.randn(5, len(CLASS_NAMES), requires_grad=True)
test_targets = torch.tensor([0, 1, 2, 0, 1], dtype=torch.long)

focal_test = FocalLoss(gamma=2.0)(test_logits, test_targets)
assert torch.isfinite(focal_test).item()
focal_test.backward()
assert test_logits.grad is not None
assert torch.isfinite(test_logits.grad).all().item()

# With gamma=0 and no alpha, focal loss must reduce to ordinary mean cross entropy.
logits_for_equivalence = torch.randn(4, 3)
targets_for_equivalence = torch.tensor([0, 2, 1, 0])
focal_gamma_zero = FocalLoss(gamma=0.0)(logits_for_equivalence, targets_for_equivalence)
ordinary_ce = nn.CrossEntropyLoss()(logits_for_equivalence, targets_for_equivalence)
torch.testing.assert_close(focal_gamma_zero, ordinary_ce)

print("Focal Loss unit checks passed.")